# FedHealth — Notebook 02: Preprocessing & Normalization
Produces the artifacts every tier reuses: the **fitted preprocessor** (median imputer + StandardScaler) and the stratified splits. Fitting happens on the training split only, so nothing leaks into val/test — and the FL clients + inference service load this exact object, guaranteeing identical feature vectors across the federation.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Module 3 Colab setup.
#
# Prerequisite:
# Clone the GitHub testing branch into /content/FedHealth-FYP before running
# these notebooks. Dataset CSVs remain in Google Drive at MyDrive/FYP.
#
# This repository stores shared files flat in "Module 3 ML/". The code below
# creates the temporary "fedhealth_ml" package layout expected by the notebooks.

from pathlib import Path
from google.colab import drive
import os
import shutil
import sys

REPO_DIR = Path('/content/FedHealth-FYP')
SOURCE_DIR = REPO_DIR / 'Module 3 ML'
assert SOURCE_DIR.exists(), (
    'Repository files are missing. Clone the GitHub testing branch first.'
)

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip -q install -r "/content/FedHealth-FYP/Module 3 ML/requirements.txt"

RUNTIME_DIR = Path('/content/fedhealth_runtime')
PACKAGE_DIR = RUNTIME_DIR / 'fedhealth_ml'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['schema.py', 'preprocess.py', 'models.py', 'rules.py']:
    shutil.copy2(SOURCE_DIR / filename, PACKAGE_DIR / filename)

(PACKAGE_DIR / '__init__.py').write_text(
    '# Temporary Colab package for Module 3 validation.\n'
)

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

DATA_DIR = '/content/drive/MyDrive/FYP'
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

for dataset_path in [ANEMIA, THAL, LEUK]:
    assert os.path.exists(dataset_path), f'Dataset not found: {dataset_path}'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp

print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))
print('All three datasets found in Google Drive.')


## 1. Load + clean (impossible values -> NaN)

In [ ]:
X, y = pp.load_anemia(ANEMIA)
print('X', X.shape, '| NaNs introduced by range-cleaning:', int(X.isna().sum().sum()))
print('classes present:', sorted(y.unique()))

## 2. Stratified train / val / test split
Stratification preserves the (very imbalanced) class ratios in each split.

In [ ]:
sp = pp.stratified_split(X, y, test_size=0.2, val_size=0.1, seed=42)
for k in ['X_train','X_val','X_test']:
    print(f'{k:8}', sp[k].shape)

## 3. Fit the preprocessor on TRAIN only, then save it

In [ ]:
import os
ART = '/content/artifacts/modelA'; os.makedirs(ART, exist_ok=True)
prep = pp.CBCPreprocessor(schema.MODEL_A_FEATURES).fit(sp['X_train'])
prep.save(f'{ART}/preprocessor.joblib')
Xtr = prep.transform(sp['X_train']); Xval = prep.transform(sp['X_val']); Xte = prep.transform(sp['X_test'])
print('scaled train', Xtr.shape, '| any NaN:', bool(np.isnan(Xtr).any()))
print('saved preprocessor ->', f'{ART}/preprocessor.joblib')

## 4. Encode labels + inspect class imbalance
The class weights below are what the ANN uses to avoid biasing toward frequent classes (you asked for balanced performance + healthy detection).

In [ ]:
from fedhealth_ml import models
ytr = pp.encode_labels(sp['y_train'], schema.MODEL_A_CLASSES)
yval = pp.encode_labels(sp['y_val'], schema.MODEL_A_CLASSES)
yte = pp.encode_labels(sp['y_test'], schema.MODEL_A_CLASSES)
cw = models.class_weights(ytr, len(schema.MODEL_A_CLASSES))
for i, c in enumerate(schema.MODEL_A_CLASSES):
    print(f'{c:34} n_train={int((ytr==i).sum()):4d}  weight={cw[i]:.2f}')

## 5. Save arrays for Notebook 03 and back them up to Drive

In [ ]:
np.savez(f'{ART}/arrays.npz', Xtr=Xtr, Xval=Xval, Xte=Xte, ytr=ytr, yval=yval, yte=yte)
DRIVE_ART = f'{DATA_DIR}/validation/module3/modelA'
os.makedirs(DRIVE_ART, exist_ok=True)
for filename in ['arrays.npz', 'preprocessor.joblib']:
    shutil.copy2(f'{ART}/{filename}', f'{DRIVE_ART}/{filename}')
print('saved locally ->', ART)
print('backed up to Drive ->', DRIVE_ART)

### Thalassemia preprocessing
Deferred until the Model B dataset decision (Notebook 04). The same `pp.load_thal` + `CBCPreprocessor(schema.MODEL_B_FEATURES)` pipeline applies to whichever dataset is chosen.